<a href="https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedabdelshaheed/flyrank_machinelearning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row = one page , in 3/2026

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features**

From `fact_content_daily_performance`:

`gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `ga4_sessions`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `ga4_pageviews`, `ga4_users`

From `dim_content`:

`content_created_date`, `keyword_hash_id`, `main_intent`, `backlinks`


**Label**

Percent change in average daily `gsc_impressions` per page between March 2026 and April 2026: `(avg_april - avg_march) / avg_march * 100`.

`down` if below -20%, `up` if above +20%, otherwise `stable`.

**Context**

`client_hash_id`, `content_hash_id`, `report_date`, `gsc_data_start`, `ga4_data_start`, `is_published`, `is_deleted`

**Excluded**

`fact_content_query_90d` table, because its fixed window (2 April to 30 June 2026) does not cover March.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import duckdb, pandas as pd
from getpass import getpass
from huggingface_hub import login
from datasets import load_dataset

TOKEN = getpass("HF token: ")
login(token=TOKEN)

con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{TOKEN}')")
SRC = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

START, END = "2026-03-01", "2026-03-31"

df = con.execute(f"""
SELECT report_date, client_hash_id, content_hash_id, gsc_data_available,
       gsc_impressions, gsc_clicks, gsc_sum_position,
       ga4_sessions, ga4_engaged_sessions, ga4_total_engagement_sec, ga4_pageviews, ga4_users

FROM read_parquet('{SRC}', hive_partitioning=true)
WHERE month = '2026-03'
  AND report_date BETWEEN DATE '{START}' AND DATE '{END}'
""").df()

print(len(df))
df.head()

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

9841378


,report_date,client_hash_id,content_hash_id,gsc_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,ga4_pageviews,ga4_users
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,20,0,67,<NA>,<NA>,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,1,0,0,<NA>,<NA>,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,125,1,616,<NA>,<NA>,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,7,0,28,<NA>,<NA>,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,11,0,25,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
print("كل الصفوف:", len(df))
print("صفوف gsc_data_available = True:", (df["gsc_data_available"] == True).sum())
print("null في الصفوف دي:", df.loc[df["gsc_data_available"] == True, ["gsc_impressions", "ga4_sessions"]].isna().sum().to_dict())
print("أصغر تاريخ:", df["report_date"].min(), "| أكبر تاريخ:", df["report_date"].max())

كل الصفوف: 9841378
صفوف gsc_data_available = True: 3611061
null في الصفوف دي: {'gsc_impressions': 0, 'ga4_sessions': 1528366}
أصغر تاريخ: 2026-03-01 00:00:00 | أكبر تاريخ: 2026-03-31 00:00:00


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.